<a href="https://colab.research.google.com/github/dmainagithub/LLMs-Lessons/blob/main/huggingface_text_classification_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification Tutorial

Note: a GPU is needed in google colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU

## 2. Import necessary commands

In [1]:
# Install dependencies
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio # -U stands for upgrade
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")



Using transformers version: 5.16.1
Using datasets version: 5.0.1
Using torch version: 2.11.0+cu128


## 3. Getting a dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

README.md:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.9kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [3]:
# What features are there
dataset.column_names

{'train': ['text', 'label']}

In [4]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [5]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [6]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f" Text: {text} | Label: {label}")


[95, 157, 226, 9, 206]
[INFO] Random samples from dataset:

 Text: Set of keys hanging on a hook by the door | Label: not_food
 Text: Vacuum cleaner stored in a closet | Label: not_food
 Text: Vintage record player spinning a vinyl record | Label: not_food
 Text: Sushi with unique toppings like seared tuna or eel sauce. | Label: food
 Text: Set of skis leaning against a wall | Label: not_food


In [7]:
range(len(dataset["train"]))

range(0, 250)

In [8]:
dataset["train"].unique("label")

['food', 'not_food']

In [9]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])


Counter({'food': 125, 'not_food': 125})

In [10]:
# Turn our dataset into a dataframe
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
173,Microscope set up on a table,not_food
93,Garden hose rolled up and ready in a yard,not_food
68,King-size bed with a white comforter inviting ...,not_food
103,"Carrots on a plate, served with a side of crea...",food
213,Chandelier casting light in a dining room,not_food
3,Wooden dresser with a mirror reflecting the room,not_food
230,Wooden cutting board with a chef's knife ready...,not_food


In [11]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

1. Tokenization (machines prefer numbers rather than words)

2. Creating a train-test split (train split for training and test split for evaluation)

In [12]:
# Create a mapping programmatically
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: idx for idx, label in id2label.items()}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [13]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
  print(idx, label)
  id2label[idx] = label

0 not_food
1 food


In [14]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample ={"text": "This is a sentence about my favorite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favorite food: honey', 'label': 1}

In [15]:
# Map our dataset labels to numbers (the whole dataset)
# With dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [16]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

{'text': ['A bowl of sliced bananas with a sprinkle of cocoa powder and a side of peanut butter',
  'A slice of pizza with a spicy buffalo chicken topping and a drizzle of ranch dressing',
  'Uniquely presented sushi roll, such as a hand roll or sushi burrito.',
  'A square slice of Sicilian-style pizza with a thick and fluffy crust',
  'Pizza with a unique topping combination of pineapple and ham'],
 'label': [1, 1, 1, 1, 1]}

### Train Test Split

* https://huggingface.co/docs/datasets/v4.8.4/process  

In [17]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [18]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Pizza with a unique topping combination of pineapple and ham',
 'label': 1}

In [19]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["train"][random_idx_test]
random_sample_test

{'text': 'Set of tea towels folded in a kitchen', 'label': 0}

## Tokenization

https://platform.openai.com/tokenizer - open ai tokenizers.
https://github.com/huggingface/tokenizers - huggingface tokenizers.
To do this locally you need to have rust installed.
* RUST is a programming language: https://rust-lang.org/
* Huggingface auto classes: https://huggingface.co/docs/transformers/en/model_doc/auto

To find all the models: https://huggingface.co/models

We will use this specific one: https://huggingface.co/distilbert/distilbert-base-uncased

** Models are often paired with tokenizers.
* Tokenizers = turn text to numbers.
* Models = find patterns in those numbers.


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True)
tokenizer

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [21]:
# Test our tokenizer
# Open ai token ids for "I love pizza" = [[40, 3047, 27941]]
tokenizer("I love pizza")

{'input_ids': [101, 1045, 2293, 10733, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

### Tokenizer vocabulary and input sequence

In [22]:
# Tokenizer vocab
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"[INFO] Number of items in our tokenizer vocab: {length_of_tokenizer_vocab}")

# Maximum sequence length the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"[INFO] Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}")

[INFO] Number of items in our tokenizer vocab: 30522
[INFO] Max tokenizer input sequence length: 512


In [29]:
# Does Nderitu occur in the vocab?
tokenizer.vocab["nderitu"]

KeyError: 'nderitu'

In [58]:
# tokenizer.vocab

In [33]:
tokenizer("Nderitu")

{'input_ids': [101, 1050, 4063, 4183, 2226, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [34]:
tokenizer.convert_ids_to_tokens(tokenizer("Nderitu").input_ids)

['[CLS]', 'n', '##der', '##it', '##u', '[SEP]']

In [35]:
# Tokenizing an emoji
tokenizer.convert_ids_to_tokens(tokenizer("👍").input_ids)

['[CLS]', '[UNK]', '[SEP]']

In [37]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

[('!', 999), ('"', 1000), ('#', 1001), ('##!', 29612), ('##"', 29613)]

In [39]:
# Get 5 random items from the vocab
import random

random.sample(sorted(tokenizer.vocab.items()), k=7)

[('feud', 13552),
 ('[unused964]', 969),
 ('inquiries', 27050),
 ('meet', 3113),
 ('1866', 7647),
 ('cardiovascular', 22935),
 ('medieval', 5781)]

## Part 5: Preparing text data for use with a model

* Preprocessing function

In [46]:
def tokenize_text(example):
  """
  Tokenize given example text and return the tokenized text.
  """
  return tokenizer(example["text"],
                   padding=True,
                   truncation=True)

In [41]:
tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [47]:
example_sample_2 = {"text": "I love my wife", "label": 0}

# Test the function
tokenize_text(example_sample_2)

{'input_ids': [101, 1045, 2293, 2026, 2564, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [49]:
long_text = "I love you very much " * 1000
len(long_text)

21000

In [51]:
tokenize_long_text = tokenize_text({"text": long_text, "label": 0})
len(tokenize_long_text["input_ids"])

512

### Tokenize our dataset

In [53]:
# Map our tokenize text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50
    })
})

In [54]:
tokenizer.all_special_tokens

['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']

In [55]:
tokenizer.all_special_ids

[100, 102, 0, 101, 103]

In [57]:
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]

for key in train_tokenized_sample.keys():
  print(f"[INFO] Key: {key}")
  print(f"Train sample: {train_tokenized_sample[key]}")
  print(f"Test sample: {test_tokenized_sample[key]}")
  print()

[INFO] Key: text
Train sample: Set of headphones placed on a desk
Test sample: A slice of pepperoni pizza with a layer of melted cheese

[INFO] Key: label
Train sample: 0
Test sample: 1

[INFO] Key: input_ids
Train sample: [101, 2275, 1997, 2132, 19093, 2872, 2006, 1037, 4624, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [101, 1037, 14704, 1997, 11565, 10698, 10733, 2007, 1037, 6741, 1997, 12501, 8808, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[INFO] Key: token_type_ids
Train sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[INFO] Key: attention_mask
Train sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0